# Phase 1-4 on Kaggle / Colab GPU

Trains `m5`, `logmel_cnn` and `effnet_b0` on CirCor (patient-level 5-fold CV, 3 seeds; `tiny_cnn` was done on the laptop), then runs the compression sweep, the trust audit and the shift analysis, using the official CinC 2022 scorer.

**Kaggle:** Session options -> Accelerator: GPU T4 x2; Internet: ON. Then *Save Version -> Save & Run All (Commit)* so it keeps running after you close the tab (12 h limit). Runs are resumable: to continue after a limit, attach the previous version's output as input and re-run.

**Code source:** cloned from https://github.com/Mustafa-bacha/pcg-quant-audit (public).

At the end download `runs.zip` from the Output tab and unzip it into the project's `runs/` folder on the laptop.

In [ ]:
import os, glob, subprocess, sys, zipfile
from pathlib import Path

ON_KAGGLE = Path('/kaggle/working').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('/content')
PROJ = WORK / 'pcg_project'
REPO = 'https://github.com/Mustafa-bacha/pcg-quant-audit.git'

if not (PROJ / 'pcg').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(PROJ)], check=True)
os.chdir(PROJ)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'captum', 'onnx', 'onnxruntime', 'torchao', 'tabulate'], check=True)
import torch, torchvision
print('project:', PROJ, '| torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# CirCor v1.0.3 from the AWS Open Data mirror (~586 MB, usually 1-3 min on Kaggle).
raw = PROJ / 'data' / 'raw'
raw.mkdir(parents=True, exist_ok=True)
pre = [p for p in glob.glob('/kaggle/input/**/training_data', recursive=True) if Path(p).is_dir()]
if pre and not (raw / 'circor-heart-sound-1.0.3').exists():
    os.symlink(Path(pre[0]).parent, raw / 'circor-heart-sound-1.0.3', target_is_directory=True)
    print('using attached dataset:', pre[0])
else:
    subprocess.run([sys.executable, 'scripts/download_circor_s3.py'], check=True)
subprocess.run([sys.executable, 'scripts/p0_verify_data.py'], check=True)

In [ ]:
# Phase 1: the FP32 grid. Edit MODELS/FOLDS/SEEDS to split across sessions if needed.
MODELS = ['m5', 'logmel_cnn', 'effnet_b0']  # tiny_cnn was run on the laptop
FOLDS = ['0', '1', '2', '3', '4']
SEEDS = ['0', '1', '2']
EPOCHS = '30'
RUNS = str(WORK / 'runs')
cmd = [sys.executable, 'scripts/p1_train.py', '--model', *MODELS, '--folds', *FOLDS, '--seeds', *SEEDS,
       '--epochs', EPOCHS, '--runs-dir', RUNS, '--workers', '2']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Phase 2: compression sweep (CPU-bound). --ort-mitigations adds the percentile/entropy/symmetric ORT arms;
# --predict-val saves validation-fold probabilities for temperature scaling in Phase 3.
cmd = [sys.executable, 'scripts/p2_quantize.py', '--model', *MODELS, '--folds', *FOLDS, '--seeds', *SEEDS,
       '--runs-dir', RUNS, '--qat-epochs', '5', '--threads', '4', '--ort-mitigations', '--predict-val']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Phase 3: trust audit (calibration, subgroups with bootstrap CIs, explanations with drift and phase metrics).
cmd = [sys.executable, 'scripts/p3_audit.py', '--model', *MODELS, '--folds', *FOLDS, '--seeds', *SEEDS,
       '--runs-dir', RUNS, '--n-boot', '2000', '--n-windows', '60', '--ig-steps', '32']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Phase 4: shift. Needs PhysioNet 2016 (~350 MB from the S3 mirror). Add '--p16-max', '900' to subsample if time is short.
subprocess.run([sys.executable, 'scripts/download_circor_s3.py', '--prefix', 'challenge-2016/1.0.0/', '--dest', 'physionet-2016', '--only', 'training-,validation/'], check=True)
cmd = [sys.executable, 'scripts/p4_shift.py', '--model', *MODELS, '--folds', *FOLDS, '--seeds', *SEEDS, '--runs-dir', RUNS]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Summarise and package results for download (runs.zip: checkpoints, ONNX files, per-recording probabilities, metrics).
import json, pandas as pd
subprocess.run([sys.executable, 'scripts/report.py', '--runs-dir', RUNS, '--out', str(WORK / 'results' / 'RESULTS.md')], check=True)
rows = []
for f in glob.glob(f'{RUNS}/*/fold*/seed*/patient_test.json'):
    p = Path(f)
    rows.append({'model': p.parts[-4], 'fold': p.parts[-3], 'seed': p.parts[-2]} | json.loads(p.read_text()))
df = pd.DataFrame(rows)
display(df.groupby('model')[['weighted_accuracy', 'auroc', 'present_recall', 'cost']].agg(['mean', 'std']).round(3))
out = WORK / 'runs.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in list(Path(RUNS).rglob('*')) + list((WORK / 'results').rglob('*')):
        if f.is_file():
            z.write(f, f.relative_to(WORK).as_posix())
print('download:', out, f'{out.stat().st_size/1e6:.1f} MB')